In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
FOLDERS = {
    "data/full-drumstick/test2-0": "No fracture hammer 1",
    "data/full-drumstick/test2-100": "100% Fracture hammer 1",
    "data/full-drumstick/test3-0": "No fracture hammer 2",
    "data/full-drumstick/test3-100": "100% Fracture hammer 2",
    "data/full-drumstick/test4-0": "No fracture hammer 3",
    "data/full-drumstick/test4-100": "100% Fracture hammer 3",
    "data/full-drumstick/test5-0": "No fracture hammer 4",
    "data/full-drumstick/test5-100": "100% Fracture hammer 4"
}

phase_data = {}

for folder, label in FOLDERS.items():

    csv_files = sorted(Path(folder).glob("*.csv"))

    all_phase = []

    for file in csv_files:

        df = pd.read_csv(file)

        phase = np.degrees(
            np.arctan2(
                df["imaginary"],
                df["real"]
            )
        )

        all_phase.append(phase)

    all_phase = np.vstack(all_phase)

    phase_data[label] = {
        "frequency": df["frequency"].values,
        "mean": np.mean(all_phase, axis=0),
        "std": np.std(all_phase, axis=0),
        "n": len(csv_files)
    }

Phase spectra

In [ ]:
plt.figure(figsize=(10,6))

for label, data in phase_data.items():

    plt.plot(
        data["frequency"],
        data["mean"],
        label=label
    )

plt.xscale("log")

plt.xlabel("Frequency (Hz)")
plt.ylabel("Phase Angle (°)")
plt.title("Phase Spectra for Hammer Fracture Experiments")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("Phase Spectra for Hammer Fracture Experiments.png")
plt.show()

Phase difference

In [ ]:
# Calculate and plot the phase difference between fractured and intact
# measurements for each hammer fracture experiment.

experiments = [
    ("Hammer experiment 1", "No fracture hammer 1", "100% Fracture hammer 1"),
    ("Hammer experiment 2", "No fracture hammer 2", "100% Fracture hammer 2"),
    ("Hammer experiment 3", "No fracture hammer 3", "100% Fracture hammer 3"),
    ("Hammer experiment 4", "No fracture hammer 4", "100% Fracture hammer 4"),
]

plt.figure(figsize=(10, 6))

for experiment_name, intact_label, fractured_label in experiments:

    intact = phase_data[intact_label]
    fractured = phase_data[fractured_label]

    # Check that both measurements use the same frequency points
    if not np.array_equal(intact["frequency"], fractured["frequency"]):
        raise ValueError(
            f"Frequency points do not match for {experiment_name}."
        )

    phase_difference = fractured["mean"] - intact["mean"]

    plt.plot(
        intact["frequency"],
        phase_difference,
        label=f"{experiment_name} (n={intact['n']})"
    )

plt.axhline(0, linewidth=1)
# plt.xscale("log")

plt.xlabel("Frequency (Hz)")
plt.ylabel("Phase Difference (°)")
plt.title("Phase angle difference between fractured and intact conditions for hammer fracture experiments")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("Phase Difference Hammer Fracture Experiments.png", dpi=300)
plt.show()


Phase at 100 kHz

In [ ]:
labels = []
means = []
errors = []

for label, data in phase_data.items():

    idx = np.argmin(
        np.abs(data["frequency"] - 100000)
    )

    labels.append(label)
    means.append(data["mean"][idx])
    errors.append(data["std"][idx])

colors = [
    "C0", "C0",
    "C1", "C1",
    "C2", "C2",
    "C3", "C3"
]

plt.figure(figsize=(10,6))

plt.bar(
    labels,
    means,
    yerr=errors,
    capsize=5,
    color=colors
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Phase Angle at 100 kHz (°)")
plt.title("Phase Angle at 100 kHz")
plt.grid()


plt.tight_layout()
plt.savefig("Bar graph Phase Angle at 100 kHz.png")
plt.show()